# Classificação de personalidade com LightGBM
Desafio Rocketseat — prever `personality_type` (Introvert, Extrovert ou Ambivert) a partir de 29 traços psicológicos, com classificação multiclasse e otimização de hiperparâmetros.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, accuracy_score, f1_score, log_loss,
                             roc_auc_score, ConfusionMatrixDisplay)

## 1. Carregamento e análise exploratória

In [ ]:
df = pd.read_csv("personality_synthetic_dataset.csv")
print(df.shape, "| nulos:", df.isna().sum().sum(), "| duplicados:", df.duplicated().sum())
print(df.personality_type.value_counts(normalize=True).round(3))
df.describe().T

In [ ]:
# relação das features com a variável-alvo: média de cada traço por tipo de personalidade
means = df.groupby("personality_type").mean(numeric_only=True).T
means["amplitude"] = means.max(axis=1) - means.min(axis=1)
means.sort_values("amplitude", ascending=False).head(10).round(2)

In [ ]:
df.hist(figsize=(16, 12), bins=20); plt.tight_layout(); plt.show()

## 2. Preparação dos dados
Não há valores ausentes, e modelos baseados em árvores não exigem normalização. O alvo é codificado com `LabelEncoder`, e a divisão treino/teste é estratificada (as classes já são balanceadas).

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df["personality_type"])
X = df.drop(columns="personality_type")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(dict(enumerate(le.classes_)))

## 3. Modelo base (LightGBM com parâmetros padrão)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
base = LGBMClassifier(objective="multiclass", random_state=42, verbose=-1)
print("Baseline CV accuracy:", cross_val_score(base, Xtr, ytr, cv=cv, scoring="accuracy").mean().round(4))

## 4. Otimização de hiperparâmetros
Busca aleatória (`RandomizedSearchCV`) otimizando o **F1 macro**, que trata as três classes com o mesmo peso.

In [ ]:
space = {
    "n_estimators": randint(100, 600),
    "learning_rate": uniform(0.01, 0.15),
    "num_leaves": randint(8, 64),
    "max_depth": [-1, 4, 6, 8],
    "min_child_samples": randint(10, 60),
    "subsample": uniform(0.6, 0.4),
    "subsample_freq": [1],
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_lambda": uniform(0, 5),
}
search = RandomizedSearchCV(base, space, n_iter=20, scoring="f1_macro", cv=3,
                            random_state=42, n_jobs=-1, verbose=1)
search.fit(Xtr, ytr)
print("Melhores parâmetros:", search.best_params_)
print("CV F1 macro (otimizado):", round(search.best_score_, 4))
best = search.best_estimator_

## 5. Avaliação no conjunto de teste

In [ ]:
pred = best.predict(Xte); proba = best.predict_proba(Xte)
print(classification_report(yte, pred, target_names=le.classes_))
print("Acurácia:", round(accuracy_score(yte, pred), 4), "| F1 macro:", round(f1_score(yte, pred, average="macro"), 4))
print("Log loss:", round(log_loss(yte, proba), 4), "| ROC AUC (ovr):", round(roc_auc_score(yte, proba, multi_class="ovr"), 4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=le.classes_, ax=ax[0], colorbar=False)
ax[0].set_title("Matriz de confusão")
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values().tail(15)
imp.plot.barh(ax=ax[1], title="Top 15 importâncias"); plt.tight_layout(); plt.show()

## 6. Principais descobertas
- Escreva aqui, com base na sua execução: quais traços mais pesam, quais classes o modelo confunde, quanto a otimização melhorou o baseline.
- Atenção: o dataset é **sintético** e as classes parecem muito separáveis. Se o baseline já ficar próximo da perfeição, comente isso: ganhos da otimização serão pequenos e o resultado não representa dados reais de personalidade.
- Próximos passos: comparar com Random Forest/CatBoost, testar `early_stopping`, adicionar ruído ou usar validação em dados reais.